# Produit scalaire induit par une matrice symétrique définie positive
## Illustration du changement d'échelle

Ce notebook illustre comment un produit scalaire défini par $\langle x, y \rangle_A = x^T A y$ (où $A$ est symétrique définie positive) équivaut à effectuer le produit scalaire euclidien standard après un changement d'échelle des variables.

In [ ]:
using LinearAlgebra
using Plots
using Printf

In [ ]:
default(
    size=(600, 600),
    dpi=150,
    legend=:best,
    grid=true,
    gridstyle=:dot,
    gridalpha=0.3
)

## 1. Exemple avec une matrice diagonale

Commençons par le cas simple d'une matrice diagonale où le changement d'échelle est direct.

In [ ]:
println("="^60)
println("EXEMPLE 1: MATRICE DIAGONALE")
println("="^60)

# Définir la matrice A (diagonale)
A_diag = [2.0  0.0;
          0.0  0.5]

println("\nMatrice A (diagonale):")
display(A_diag)

In [ ]:
# Vérifier qu'elle est définie positive
λ_diag = eigvals(A_diag)
println("\nValeurs propres: ", λ_diag)
println("Définie positive: ", all(λ_diag .> 0))

In [ ]:
# Calculer A^(1/2)
A_sqrt_diag = sqrt(A_diag)
println("\nMatrice A^(1/2):")
display(A_sqrt_diag)

# Vérifier que (A^(1/2))^T * A^(1/2) = A
println("\nVérification: (A^(1/2))' * A^(1/2) ≈ A:")
display(A_sqrt_diag' * A_sqrt_diag)

### Calcul du produit scalaire

Vérifions que $\langle x, y \rangle_A = \langle \tilde{x}, \tilde{y} \rangle$ où $\tilde{x} = A^{1/2} x$.

In [ ]:
# Exemple de calcul avec deux vecteurs
x = [1.0, 2.0]
y = [3.0, 1.0]

println("\n" * "="^60)
println("Calcul du produit scalaire")
println("="^60)
println("x = ", x)
println("y = ", y)

# Produit scalaire induit par A
prod_A = x' * A_diag * y
println("\n⟨x, y⟩_A = x' * A * y = ", prod_A)

# Changement de variables
x_tilde = A_sqrt_diag * x
y_tilde = A_sqrt_diag * y
println("\nx̃ = A^(1/2) * x = ", x_tilde)
println("ỹ = A^(1/2) * y = ", y_tilde)

# Produit scalaire euclidien standard
prod_euclidien = x_tilde' * y_tilde
println("\n⟨x̃, ỹ⟩ = x̃' * ỹ = ", prod_euclidien)

println("\nLes deux produits sont égaux: ", isapprox(prod_A, prod_euclidien))

### Visualisation

En notant
$$
A = \begin{pmatrix}
a_{11} & 0 & \ldots & 0 \\
0 & a_{22} & \ldots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \ldots & a_{nn}
\end{pmatrix},
$$
nous avons
$$
\tilde{x} =
\begin{pmatrix}
\sqrt{a_{11}} x_1 \\ \sqrt{a_{22}} x_2 \\ \vdots \\ \sqrt{a_{nn}} x_n
\end{pmatrix}.
$$
Nous faisons donc un changement d'échelle des composantes de $x$.!

Montrons comment l'ellipse $\{x : \langle x, x \rangle_A = 1\}$ devient un cercle unité après transformation.

In [ ]:
# Générer des points sur le cercle unité pour ⟨x, x⟩_A = 1
θ = range(0, 2π, length=100)
circle_A = zeros(2, 100)
A_inv = inv(A_diag)
A_inv_sqrt = sqrt(A_inv)

for i in 1:100
    v = [cos(θ[i]), sin(θ[i])]
    circle_A[:, i] = A_inv_sqrt * v
end

# Points transformés
circle_transformed = A_sqrt_diag * circle_A;

In [ ]:
p1 = plot(circle_A[1, :], circle_A[2, :], 
    label="⟨x, x⟩_A = 1 (espace original)",
    aspect_ratio=:equal, 
    linewidth=2,
    title="Matrice Diagonale",
    xlabel="x₁", ylabel="x₂",
    legend=:topright,
    size=(600, 600))

plot!(p1, circle_transformed[1, :], circle_transformed[2, :], 
    label="Cercle unité (après transformation)",
    linewidth=2, linestyle=:dash)

# Tracer les vecteurs x et y
quiver!(p1, [0], [0], quiver=([x[1]], [x[2]]), 
    color=:red, linewidth=2, label="x")
quiver!(p1, [0], [0], quiver=([y[1]], [y[2]]), 
    color=:blue, linewidth=2, label="y")

# Tracer les vecteurs transformés
quiver!(p1, [0], [0], quiver=([x_tilde[1]], [x_tilde[2]]), 
    color=:red, linewidth=2, linestyle=:dot, label="x̃")
quiver!(p1, [0], [0], quiver=([y_tilde[1]], [y_tilde[2]]), 
    color=:blue, linewidth=2, linestyle=:dot, label="ỹ")

display(p1)

## 2. Exemple avec une matrice non diagonale

Pour une matrice non diagonale, la transformation implique une rotation en plus du changement d'échelle.

Soit $A \succ 0$. Il existe plusieurs méthodes pour calculer $A^{1/2}$. L'une d'elles est la décomposition spectrale
$$
A = Q\Lambda Q^T,
$$
où $\Lambda$ est la matrice des valeurs propres de $A$ ($\Lambda = \text{diag}(\lambda_i)$) et $Q$ est la matrice des vecteurs propres associés.

$Q$ est une matrice orthogonale, et $det(Q) = \pm 1$, et nous pouvons imposer $det(Q) = +1$ en changeant au besoin l'orientation d'un vecteur propre, i.e. en multipliant une colonne par -1. Si $det(Q) = +1$, $Q$ est une matrice de rotation.

Dès lors, la transformation $\tilde{x} = A^{1/2} x$ se décompose en trois étapes :
1. **Rotation** par $Q^T$ (vers les axes principaux)
2. **Changement d'échelle** par $\Lambda^{1/2}$
3. **Rotation inverse** par $Q$ (retour aux axes originaux)

In [ ]:
println("\n\n" * "="^60)
println("EXEMPLE 2: MATRICE NON DIAGONALE")
println("="^60)

# Définir une matrice symétrique définie positive (non diagonale)
A_nondiag = [3.0  1.0;
             1.0  2.0]

println("\nMatrice A (non diagonale):")
display(A_nondiag)

# Vérifier qu'elle est symétrique
println("\nSymétrique: ", issymmetric(A_nondiag))

In [ ]:
# Valeurs et vecteurs propres
λ_nondiag, Q = eigen(A_nondiag)
println("\nValeurs propres: ", λ_nondiag)
println("Définie positive: ", all(λ_nondiag .> 0))
println("\nMatrice des vecteurs propres Q:")
display(Q)

$Q$ est-elle une matrice de rotation?

In [ ]:
det(Q)

Le déterminant de $Q$ vaut -1, or nous voudrions avoir +1. Nous allons multiplier une colonne de $Q$ par -1, par exemple la deuxième colonne.

In [ ]:
Q[:,2] *= -1
det(Q)

Construisons $A^(1/2)$.

In [ ]:
# Calculer A^(1/2) via décomposition spectrale
Λ_sqrt = Diagonal(sqrt.(λ_nondiag))
A_sqrt_nondiag = Q * Λ_sqrt * Q'

println("\nMatrice A^(1/2):")
display(A_sqrt_nondiag)

# Vérification
println("\nVérification: (A^(1/2))' * A^(1/2) ≈ A:")
println("Erreur maximale: ", maximum(abs.(A_sqrt_nondiag' * A_sqrt_nondiag - A_nondiag)))

### Visualisation

In [ ]:
# Calcul avec les mêmes vecteurs
println("\n" * "="^60)
println("Calcul du produit scalaire")
println("="^60)
println("x = ", x)
println("y = ", y)

# Produit scalaire induit par A
prod_A_nondiag = x' * A_nondiag * y
println("\n⟨x, y⟩_A = x' * A * y = ", prod_A_nondiag)

# Changement de variables
x_tilde_nondiag = A_sqrt_nondiag * x
y_tilde_nondiag = A_sqrt_nondiag * y
println("\nx̃ = A^(1/2) * x = ", x_tilde_nondiag)
println("ỹ = A^(1/2) * y = ", y_tilde_nondiag)

# Produit scalaire euclidien standard
prod_euclidien_nondiag = x_tilde_nondiag' * y_tilde_nondiag
println("\n⟨x̃, ỹ⟩ = x̃' * ỹ = ", prod_euclidien_nondiag)

println("\nLes deux produits sont égaux: ", isapprox(prod_A_nondiag, prod_euclidien_nondiag))

In [ ]:
# Générer l'ellipse ⟨x, x⟩_A = 1
circle_A_nondiag = zeros(2, 100)
A_inv_nondiag = inv(A_nondiag)
A_inv_sqrt_nondiag = sqrt(A_inv_nondiag)

for i in 1:100
    v = [cos(θ[i]), sin(θ[i])]
    circle_A_nondiag[:, i] = A_inv_sqrt_nondiag * v
end

# Points transformés (doivent former un cercle)
circle_transformed_nondiag = A_sqrt_nondiag * circle_A_nondiag

In [ ]:
p2 = plot(circle_A_nondiag[1, :], circle_A_nondiag[2, :], 
    label="⟨x, x⟩_A = 1 (espace original)",
    aspect_ratio=:equal, 
    linewidth=2,
    title="Matrice Non Diagonale",
    xlabel="x₁", ylabel="x₂",
    legend=:topright,
    size=(600, 600))

plot!(p2, circle_transformed_nondiag[1, :], circle_transformed_nondiag[2, :], 
    label="Cercle unité (après transformation)",
    linewidth=2, linestyle=:dash)

# Tracer les axes principaux (vecteurs propres)
for i in 1:2
    v = Q[:, i] * sqrt(1/λ_nondiag[i])
    quiver!(p2, [0], [0], quiver=([v[1]], [v[2]]), 
        color=:gray, linewidth=1.5, linestyle=:dashdot, 
        label=(i==1 ? "Axes principaux" : ""))
end

# Tracer les vecteurs x et y
quiver!(p2, [0], [0], quiver=([x[1]], [x[2]]), 
    color=:red, linewidth=2, label="x")
quiver!(p2, [0], [0], quiver=([y[1]], [y[2]]), 
    color=:blue, linewidth=2, label="y")

# Tracer les vecteurs transformés
quiver!(p2, [0], [0], quiver=([x_tilde_nondiag[1]], [x_tilde_nondiag[2]]), 
    color=:red, linewidth=2, linestyle=:dot, label="x̃")
quiver!(p2, [0], [0], quiver=([y_tilde_nondiag[1]], [y_tilde_nondiag[2]]), 
    color=:blue, linewidth=2, linestyle=:dot, label="ỹ")

display(p2)

In [ ]:
println("\n\n" * "="^60)
println("DÉCOMPOSITION DU CHANGEMENT DE VARIABLES")
println("="^60)

println("\nPour la matrice non diagonale, le changement x̃ = A^(1/2) * x se décompose en:")
println("1. Rotation par Q'")
println("2. Changement d'échelle par Λ^(1/2)")
println("3. Rotation par Q")

# Exemple avec x
println("\nApplication à x = ", x)

# Étape 1: Rotation
x_step1 = Q' * x
println("1. Après rotation Q': ", x_step1)

# Étape 2: Changement d'échelle
x_step2 = Λ_sqrt * x_step1
println("2. Après échelle Λ^(1/2): ", x_step2)

# Étape 3: Rotation inverse
x_step3 = Q * x_step2
println("3. Après rotation Q: ", x_step3)

println("\nVérification: x̃ = ", x_tilde_nondiag)
println("Égal au résultat direct: ", isapprox(x_step3, x_tilde_nondiag))

In [ ]:
# Visualisation des étapes
p3 = scatter([x[1]], [x[2]], 
    label="x original", 
    markersize=8, 
    color=:red,
    aspect_ratio=:equal,
    title="Décomposition de la transformation",
    xlabel="x₁", ylabel="x₂",
    xlim=(-2, 4), ylim=(-1, 4),
    size=(600, 600))

quiver!(p3, [0], [0], quiver=([x[1]], [x[2]]), 
    color=:red, linewidth=2, alpha=0.5)

scatter!(p3, [x_step1[1]], [x_step1[2]], 
    label="Après Q'", markersize=8, color=:orange)
quiver!(p3, [0], [0], quiver=([x_step1[1]], [x_step1[2]]), 
    color=:orange, linewidth=2, alpha=0.5)

scatter!(p3, [x_step2[1]], [x_step2[2]], 
    label="Après Λ^(1/2)", markersize=8, color=:green)
quiver!(p3, [0], [0], quiver=([x_step2[1]], [x_step2[2]]), 
    color=:green, linewidth=2, alpha=0.5)

scatter!(p3, [x_step3[1]], [x_step3[2]], 
    label="x̃ final", markersize=8, color=:blue)
quiver!(p3, [0], [0], quiver=([x_step3[1]], [x_step3[2]]), 
    color=:blue, linewidth=2)

display(p3)

## 4. Application : Optimisation par plus forte pente

Considérons le problème d'optimisation quadratique :

$$\min_x \frac{1}{2} x^T A x + b^T x$$

où $A$ est symétrique définie positive.

### Point clé

La **plus forte pente** dépend de la métrique utilisée :
- Avec la métrique euclidienne : $-\nabla f(x) = -(Ax + b)$
- Avec la métrique induite par $A$ : $-A^{-1}\nabla f(x) = -A^{-1}(Ax + b)$

Utiliser la métrique $A$ permet de converger en **une seule itération** !

### Démonstration : La plus forte pente dépend de la métrique

#### Définition de la plus forte pente

La **direction de plus forte pente** en un point $x$ est la direction $d$ (de norme 1 selon une certaine métrique) qui maximise la décroissance de la fonction, c'est-à-dire qui minimise la dérivée directionnelle :

$$d^* = \arg\min_{\|d\|=1} \nabla f(x)^T d$$

où la norme $\|\cdot\|$ dépend de la métrique choisie.

---

#### Cas 1 : Métrique euclidienne

**Métrique** : $\|d\|_2 = \sqrt{d^T d}$

**Problème d'optimisation** :
$$\min_{d} \nabla f(x)^T d \quad \text{sujet à} \quad d^T d = 1$$

**Solution par multiplicateurs de Lagrange** :

Lagrangien : $\mathcal{L}(d, \lambda) = \nabla f(x)^T d + \lambda(d^T d - 1)$

Conditions de stationnarité :
$$\nabla_d \mathcal{L} = \nabla f(x) + 2\lambda d = 0$$

$$\Rightarrow d = -\frac{1}{2\lambda} \nabla f(x)$$

En utilisant la contrainte $d^T d = 1$ :
$$\frac{1}{4\lambda^2} \nabla f(x)^T \nabla f(x) = 1 \quad \Rightarrow \quad \lambda = \pm \frac{1}{2}\|\nabla f(x)\|_2$$

Pour minimiser (descente), $\lambda = +\frac{1}{2}\|\nabla f(x)\|_2$, donc :

$$\boxed{d^*_{\text{eucl}} = -\frac{\nabla f(x)}{\|\nabla f(x)\|_2} = -\frac{Ax + b}{\|Ax + b\|_2}}$$

**Direction non normalisée** : $-\nabla f(x) = -(Ax + b)$

---

#### Cas 2 : Métrique induite par $A$

**Métrique** : $\|d\|_A = \sqrt{d^T A d}$ avec produit scalaire $\langle d, d \rangle_A = d^T A d$

**Problème d'optimisation** :
$$\min_{d} \nabla f(x)^T d \quad \text{sujet à} \quad d^T A d = 1$$

**Solution par multiplicateurs de Lagrange** :

Lagrangien : $\mathcal{L}(d, \lambda) = \nabla f(x)^T d + \lambda(d^T A d - 1)$

Conditions de stationnarité :
$$\nabla_d \mathcal{L} = \nabla f(x) + 2\lambda A d = 0$$

$$\Rightarrow Ad = -\frac{1}{2\lambda} \nabla f(x)$$

$$\Rightarrow d = -\frac{1}{2\lambda} A^{-1} \nabla f(x)$$

En utilisant la contrainte $d^T A d = 1$ :
$$\frac{1}{4\lambda^2} \nabla f(x)^T A^{-1} A A^{-1} \nabla f(x) = 1$$

$$\frac{1}{4\lambda^2} \nabla f(x)^T A^{-1} \nabla f(x) = 1 \quad \Rightarrow \quad \lambda = \pm \frac{1}{2}\|\nabla f(x)\|_{A^{-1}}$$

Pour minimiser (descente), $\lambda = +\frac{1}{2}\|\nabla f(x)\|_{A^{-1}}$, donc :

$$\boxed{d^*_A = -\frac{A^{-1}\nabla f(x)}{\|A^{-1}\nabla f(x)\|_A} = -\frac{A^{-1}(Ax + b)}{\|A^{-1}(Ax + b)\|_A}}$$

**Direction non normalisée** : $-A^{-1}\nabla f(x) = -A^{-1}(Ax + b)$

---

#### Vérification de la norme

Vérifions que $d^*_A$ a bien une norme $A$ égale à 1 :

$$\|d^*_A\|_A^2 = d^{*T}_A A d^*_A = \frac{(A^{-1}\nabla f)^T A (A^{-1}\nabla f)}{\|(A^{-1}\nabla f)\|_A^2}$$

$$= \frac{\nabla f^T A^{-1} A A^{-1} \nabla f}{\nabla f^T A^{-1} A A^{-1} \nabla f} = \frac{\nabla f^T A^{-1} \nabla f}{\nabla f^T A^{-1} \nabla f} = 1 \quad \checkmark$$

---

#### Interprétation géométrique

Dans l'espace transformé $\tilde{x} = A^{1/2}x$, la métrique induite par $A$ devient euclidienne :

$$\langle d, d \rangle_A = d^T A d = (A^{1/2}d)^T (A^{1/2}d) = \|\tilde{d}\|_2^2$$

**La direction de plus forte pente dans la métrique $A$ correspond donc à la direction de plus forte pente euclidienne dans l'espace transformé.**

C'est pourquoi, dans l'espace transformé où les courbes de niveau sont circulaires, le gradient A-normé pointe directement vers le centre (l'optimum).

---

#### Conclusion

Les directions de plus forte pente dépendent intrinsèquement de la métrique choisie :

| Métrique | Direction normalisée | Direction non normalisée |
|----------|---------------------|-------------------------|
| Euclidienne | $-\dfrac{\nabla f(x)}{\|\nabla f(x)\|_2}$ | $-\nabla f(x) = -(Ax + b)$ |
| Induite par $A$ | $-\dfrac{A^{-1}\nabla f(x)}{\|A^{-1}\nabla f(x)\|_A}$ | $-A^{-1}\nabla f(x) = -A^{-1}(Ax + b)$ |

**Propriété remarquable** : Pour le problème quadratique $f(x) = \frac{1}{2}x^T A x + b^T x$, la direction $d = -A^{-1}(Ax + b)$ avec un pas $\alpha = 1$ (pas optimal) conduit directement à l'optimum :

$$x_{\text{new}} = x - A^{-1}(Ax + b) = x - A^{-1}Ax - A^{-1}b = -A^{-1}b = x^*$$

Ceci explique la **convergence en une seule itération** avec la métrique $A$ ! $\quad \blacksquare$

In [ ]:
println("\n\n" * "="^60)
println("EXEMPLE 4: OPTIMISATION PAR PLUS FORTE PENTE")
println("="^60)

# Définir un problème d'optimisation non trivial
A_opt = [5.0  2.0;
         2.0  3.0]
b_opt = [1.0, -2.0]

println("\nProblème: min 1/2 x'Ax + b'x")
println("\nMatrice A:")
display(A_opt)
println("\nVecteur b: ", b_opt)

# Solution exacte
x_star = -A_opt\b_opt
println("\nSolution exacte x* = -A^(-1)b: ", x_star)

# Valeur optimale
f_star = 0.5 * x_star' * A_opt * x_star + b_opt' * x_star
println("Valeur optimale f(x*): ", f_star)

In [ ]:
# Fonction objectif et gradient
f(x) = 0.5 * x' * A_opt * x + b_opt' * x
∇f(x) = A_opt * x + b_opt

# Point de départ
x0 = [2.0, 3.0]
println("\nPoint de départ x0: ", x0)
println("f(x0): ", f(x0))
println("Gradient ∇f(x0): ", ∇f(x0))

### Méthode 1 : Descente selon le gradient euclidien

Direction de descente : $d = -\nabla f(x) = -(Ax + b)$

Le pas optimal s'obtient en résolvant
$$
\min_{\alpha} f(x+\alpha d),
$$
donnant comme condition d'optimalité
$$
\frac{d}{d\alpha} f(x+\alpha d) = 0.
$$
Nous avons
\begin{align*}
\frac{d}{d\alpha} f(x+\alpha d) &= \nabla_x f(x+\alpha d)^T d \\
& = (A(x+\alpha d)+b)^T d \\
& = (\nabla_x f(x) + \alpha d)^Td.
\end{align*}
Par conséquent,
$$
\alpha^* = -\frac{\nabla_x f(x)^Td}{d^TAd}.
$$

Le pas optimal le long de cette direction vaut dès lors
$$
\alpha^* = \frac{d^T d}{d^T A d}.
$$

In [ ]:
println("\n" * "-"^60)
println("MÉTHODE 1: Descente selon gradient euclidien")
println("-"^60)

# Descente de gradient standard avec pas optimal
max_iter = 20
x_eucl = copy(x0)
history_eucl = [copy(x_eucl)]
f_history_eucl = [f(x_eucl)]

for k in 1:max_iter
    grad = ∇f(x_eucl)
    
    # Direction: gradient négatif (métrique euclidienne)
    d = -grad
    
    # Pas optimal le long de d
    α = (d' * d) / (d' * A_opt * d)
    
    # Mise à jour
    x_eucl = x_eucl + α * d
    
    push!(history_eucl, copy(x_eucl))
    push!(f_history_eucl, f(x_eucl))
    
    if norm(grad) < 1e-10
        println("Convergence atteinte à l'itération ", k)
        break
    end
end

println("\nNombre d'itérations: ", length(history_eucl) - 1)
println("Solution finale: ", x_eucl)
println("Erreur: ", norm(x_eucl - x_star))

### Méthode 2 : Descente selon le gradient A-normé (métrique A)

Direction de descente : $d = -A^{-1}\nabla f(x) = -A^{-1}(Ax + b)$

Cette direction est la **plus forte pente** selon la métrique induite par $A$ !

Comme plus haut, la longeur de pas optimale vaut
$$
\alpha^* = -\frac{\nabla_x f(x)^Td}{d^TAd}.
$$

Dès lors
$$
\alpha^* = -\frac{-\nabla_x f(x)^TA^{-1}\nabla f(x)}{-(A^{-1}\nabla f(x))^TA(-A^{-1}\nabla f(x))}
= \frac{\nabla_x f(x)^TA^{-1}\nabla f(x)}{\nabla f(x))^TA^{-1}\nabla f(x)} = 1.
$$

In [ ]:
println("\n" * "-"^60)
println("MÉTHODE 2: Descente selon gradient A-normé")
println("-"^60)

x_A = copy(x0)
history_A = [copy(x_A)]
f_history_A = [f(x_A)]

# Une seule itération suffit!
grad = ∇f(x_A)

# Direction: plus forte pente selon métrique A
d = -A_opt\ grad

println("\nDirection euclidienne: -∇f = ", -grad)
println("Direction A-normée: -A^(-1)∇f = ", d)

# Pas optimal (vaut 1 dans ce cas!)
α = 1

# Mise à jour
x_A = x_A + α * d
push!(history_A, copy(x_A))
push!(f_history_A, f(x_A))

println("\nNombre d'itérations: 1")
println("Solution finale: ", x_A)
println("Erreur: ", norm(x_A - x_star))
println("\n✓ Convergence en UNE SEULE itération!")

In [ ]:
# Visualisation des trajectoires
println("\n" * "="^60)
println("VISUALISATION DES TRAJECTOIRES")
println("="^60)

# Créer une grille pour les courbes de niveau
x1_range = range(-1, 3, length=100)
x2_range = range(-1, 4, length=100)
Z = [f([x1, x2]) for x2 in x2_range, x1 in x1_range]

p_opt = contour(x1_range, x2_range, Z, 
    levels=20,
    aspect_ratio=:equal,
    title="Comparaison des méthodes de descente",
    xlabel="x₁", ylabel="x₂",
    legend=:topright,
    size=(800, 800),
    color=:viridis)

# Tracer l'optimum
scatter!(p_opt, [x_star[1]], [x_star[2]], 
    markersize=10, color=:red, marker=:star,
    label="Optimum x*")

# Tracer le point de départ
scatter!(p_opt, [x0[1]], [x0[2]], 
    markersize=8, color=:black, marker=:circle,
    label="Point de départ x₀")

# Trajectoire gradient euclidien
x_eucl_array = hcat(history_eucl...)'
plot!(p_opt, x_eucl_array[:, 1], x_eucl_array[:, 2],
    linewidth=2, color=:blue, marker=:o, markersize=4,
    label="Gradient euclidien ($(length(history_eucl)-1) iter.)")

# Trajectoire gradient A-normé
x_A_array = hcat(history_A...)'
plot!(p_opt, x_A_array[:, 1], x_A_array[:, 2],
    linewidth=3, color=:green, marker=:square, markersize=6,
    label="Gradient A-normé (1 iter.)")

display(p_opt)

In [ ]:
# Calculer erreurs avec seuil minimum
errors_eucl_log = max.(f_history_eucl .- f_star, 1e-16)
errors_A_log = max.(f_history_A .- f_star, 1e-16)

p_conv = plot(0:length(f_history_eucl)-1, errors_eucl_log,
    yscale=:log10,
    linewidth=2, marker=:o, color=:blue,
    label="Gradient euclidien",
    xlabel="Itération",
    ylabel="f(xₖ) - f(x*)",
    title="Convergence",
    ylims=(1e-16, 1e2),        # ← CLÉ : limites explicites
    yticks=10.0 .^ (-16:3:2))  # ← CLÉ : graduations explicites

plot!(p_conv, 0:length(f_history_A)-1, errors_A_log,
    linewidth=3, marker=:square, color=:green,
    label="Gradient A-normé")

### Interprétation géométrique

Dans l'espace transformé par $\tilde{x} = A^{1/2}x$ :
- Les courbes de niveau deviennent **circulaires**
- Le gradient A-normé devient le gradient euclidien standard
- La direction de plus forte descente pointe directement vers le centre